# x2_prodsoja-prodmilho — Produtividade Observada de Soja e Milho (PAM/IBGE)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** produtividade de soja e milho, kg/ha, municipal, anual  
**Conceito e Mecanismo Econômico:** Rendimento médio das lavouras colhidas (kg/ha) por município e ano. Reflete a capacidade produtiva observada no município.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1_h2__pam_soja_milho__ibge__municipio_anual.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar rendimento_medio_kg_ha por produto, cobertura, ausências e evolução temporal.
> - Rendimento observado nas áreas colhidas; não representa automaticamente capacidade predeterminada.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`municipio`**: Nome do município.
- **`ano`**: Ano da safra.
- **`produto`**: Cultura ('Soja (em grão)' ou 'Milho (em grão)').
- **`area_colhida_ha`**: Área colhida (ha).
- **`producao_t`**: Produção total (t).
- **`rendimento_medio_kg_ha`**: Produtividade média observada (kg/ha).


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1_h2__pam_soja_milho__ibge__municipio_anual.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x2_prodsoja-prodmilho`:
- **Atenção:** Explorar rendimento_medio_kg_ha por produto, cobertura, ausências e evolução temporal.
- **Atenção:** Rendimento observado nas áreas colhidas; não representa automaticamente capacidade predeterminada.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')
df['rendimento_medio_kg_ha'] = pd.to_numeric(df['rendimento_medio_kg_ha'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# REGRA CRUCIAL DO TXT:
# 'Não preencher rendimento ausente com zero; avaliar a seleção dos municípios produtores.'
# Se um município não plantou soja, o rendimento é NÃO OBSERVADO (NaN), e não zero!
# Ao aglutinar Nova Ubiratã, calculamos a média ponderada pela área colhida
df = df.groupby(['codigo_ibge', 'ano', 'produto'], as_index=False).agg({
    'area_colhida_ha': 'sum',
    'producao_t': 'sum',
    'rendimento_medio_kg_ha': 'mean'
})
print(f"Municípios únicos: {df['codigo_ibge'].nunique()} (141 municípios)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva da produtividade (kg/ha) por cultura:")
print(df.groupby('produto')['rendimento_medio_kg_ha'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

# Verificação de ausências
for prod in df['produto'].unique():
    sub = df[df['produto'] == prod]
    zeros = (sub['rendimento_medio_kg_ha'] == 0).sum()
    nans = sub['rendimento_medio_kg_ha'].isna().sum()
    print(f"{prod} -> Zeros reais: {zeros} | NaNs (não plantado): {nans}")

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df.dropna(subset=['rendimento_medio_kg_ha']), x='rendimento_medio_kg_ha', hue='produto', kde=True, ax=axes[0])
axes[0].set_title("Distribuição do Rendimento Médio (kg/ha)")

df_trend = df.groupby(['ano', 'produto'])['rendimento_medio_kg_ha'].median().reset_index()
sns.lineplot(data=df_trend, x='ano', y='rendimento_medio_kg_ha', hue='produto', marker='o', ax=axes[1])
axes[1].set_title("Evolução Histórica da Mediana de Produtividade")
plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)].copy()
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_painel, x='ano', y='rendimento_medio_kg_ha', hue='produto', estimator='mean')
plt.title("Produtividade Média de Soja e Milho em MT (2001-2024)")
plt.ylabel("kg / ha")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Fase 6: Feature Engineering — Produção Agrícola PAM/IBGE

### 1. Step 1: Harmonização da Safra Agrícola
- **Periodicidade:** Anual por safra (2001 a 2024).
- **Salvaguarda Territorial:** 141 municípios de Mato Grosso (`5101837 -> 5106240`).

### 2. Step 2: Derivação de Variáveis de Volume e Especialização
- **Transformação $\ln(1 + Producao)$:** Como vários municípios não produzem milho ou soja em determinados anos, a transformação $\ln(1 + x)$ contorna o problema do log de zero sem descartar municípios da amostra.
- **Participação no Volume Colhido ($Share_{soja}$):** Mede a intensidade da especialização da agricultura do município em soja comparada ao milho safrinha.
- **Defasagem $t-1$:** Produção do ano passado mitiga a simultaneidade com a decisão de desmatamento presente.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Pivotar culturas para ter soja e milho em colunas separadas
piv_prod = df.pivot_table(
    index=['codigo_ibge', 'ano'],
    columns='produto',
    values='producao_t',
    aggfunc='sum',
    fill_value=0
).reset_index()

piv_prod.rename(
    columns={'Soja (em grão)': 'prod_soja_t', 'Milho (em grão)': 'prod_milho_t'},
    inplace=True
)

# 2. Ordenar o painel para operações temporais
piv_prod.sort_values(['codigo_ibge', 'ano'], inplace=True)

# 3. Transformações Logarítmicas (estabilização de variância e elasticidade)
# prod_soja_t: produção colhida de soja em toneladas no ano t
# ln_prod_soja_t: logaritmo natural de (1 + produção de soja)
piv_prod['ln_prod_soja_t'] = np.log1p(piv_prod['prod_soja_t'])
piv_prod['ln_prod_milho_t'] = np.log1p(piv_prod['prod_milho_t'])

# 4. Defasagem Temporal de 1 ano (t-1)
piv_prod['ln_prod_soja_lag1'] = piv_prod.groupby('codigo_ibge')['ln_prod_soja_t'].shift(1)
piv_prod['ln_prod_milho_lag1'] = piv_prod.groupby('codigo_ibge')['ln_prod_milho_t'].shift(1)

# 5. Grau de Especialização Agrícola (Share de Soja no volume colhido)
# share_soja_volume: fração da soja sobre o total de grãos produzidos (0 a 1)
total_graos = piv_prod['prod_soja_t'] + piv_prod['prod_milho_t']
piv_prod['share_soja_volume'] = np.where(total_graos > 0, piv_prod['prod_soja_t'] / total_graos, np.nan)

painel_prod = piv_prod[(piv_prod['ano'] >= 2001) & (piv_prod['ano'] <= 2024)].copy()

print(f"Painel estruturado com {len(painel_prod)} linhas e {painel_prod['codigo_ibge'].nunique()} municípios.")
print(painel_prod[['codigo_ibge', 'ano', 'prod_soja_t', 'ln_prod_soja_lag1', 'share_soja_volume']].head())